# Demo 09a — Mixtures and PCA

**Class 09 · B1** (slides: `slides/09_blind_signal_separation.md`)

The running example of the class: **two source signals mixed into two microphones**.
We look at the mixtures, then at PCA: what it finds (variance directions, not sources),
how a rank-1 reconstruction drops the second signal, and where the scree plot comes from.
The last sections move PCA to digits (components as images, reconstruction) and to spambase
(a scree over 57 features), with two failure cases no slide shows.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from faa import bss, data, plotlib

warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(20)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
pl.Config.set_tbl_width_chars(150)
plotlib.setup(module="09")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (plotlib.BLUE, plotlib.ORANGE, plotlib.GREEN, plotlib.RED, plotlib.SKY, plotlib.PURPLE, plotlib.GREY)
SEED = 0


def slide(title: str) -> None:
    """Every cell that prints a number of a slide starts with this line."""
    print(f"--- slide: {title} ---")


def show(frame: pl.DataFrame, digits: int = 3) -> None:
    print(frame.with_columns(pl.col(pl.Float64).round(digits)))

## The running example: two microphones, two sources

Two sines (3 Hz and a 7 Hz chirp) mixed by $A = \begin{pmatrix}0.9 & 0.4\\0.7 & 1.1\end{pmatrix}$,
sensor noise 0.05. This figure is the opening frame of the deck: sources on the left, mixtures on the right.

In [ ]:
X, S, A = bss.mixtures(seed=SEED)
n = X.shape[1]
t = np.arange(n) / n
f, ax = plt.subplots(2, 2, figsize=(plotlib.FULL, 2.3), sharex=True)
for j, (sig, col, name) in enumerate([(S[0], BLUE, "source 1 (3 Hz sine)"), (S[1], ORANGE, "source 2 (chirp)")]):
    ax[j, 0].plot(t, sig, color=col, lw=0.9)
    ax[j, 0].set_ylabel(f"$s_{j+1}$")
    ax[j, 0].set_title(name, fontsize=7.5)
for j in range(2):
    ax[j, 1].plot(t, X[j], color=GREY, lw=0.9)
    ax[j, 1].set_ylabel(f"$x_{j+1}$")
ax[0, 1].set_title("microphone 1: 0.9 s₁ + 0.4 s₂", fontsize=7.5)
ax[1, 1].set_title("microphone 2: 0.7 s₁ + 1.1 s₂", fontsize=7.5)
ax[1, 0].set_xlabel("time (s)")
ax[1, 1].set_xlabel("time (s)")
plotlib.save(f, "p1_mixtures")
print(f"A =\n{A}")
print("correlation of x1, x2:", f"{np.corrcoef(X)[0, 1]:.2f}")

## Slide: What PCA finds on the mixtures

PCA on the 2-D cloud of microphone samples finds the **variance directions**, not the sources.
The recovered signals are the projections; their correlation with the truth is far from 1.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2).fit(X.T)
P = pca.components_  # rows are the PCs, in the (samples x features) convention of X.T
f, ax = plt.subplots(figsize=(plotlib.COL, 2.1))
ax.scatter(X[0], X[1], s=3, color=GREY, alpha=0.5)
for j, (v, col) in enumerate([(pca.components_[0], BLUE), (pca.components_[1], ORANGE)]):
    ax.annotate("", xy=(v * 1.8), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=col, lw=1.6))
    ax.text(v[0] * 2.0, v[1] * 2.0, f"PC{j+1}", color=col, fontsize=7, ha="center")
ax.set_xlabel("microphone 1")
ax.set_ylabel("microphone 2")
plotlib.save(f, "p1_pca_axes")
print(f"explained variance ratio: {pca.explained_variance_ratio_.round(3)}")
print(f"PC axes (rows):\n{pca.components_.round(3)}")
print(f"true mixing A (columns are the source directions):\n{A.round(3)}")

## Slide: PCA separates the mixtures badly

The PC projections, matched to the true sources: correlations around 0.5-0.8, not the sources.
(The slide quotes these numbers; demo 09b shows ICA fixing them.)

In [ ]:
P = pca.transform(X.T).T
matched, corr = bss.match(S, P)
f, ax = plt.subplots(2, 2, figsize=(plotlib.FULL, 2.3), sharex=True)
for j in range(2):
    ax[j, 0].plot(t, S[j], color=(BLUE, ORANGE)[j], lw=0.9)
    ax[j, 0].set_ylabel(f"$s_{j+1}$ true")
    ax[j, 1].plot(t, matched[j], color=GREY, lw=0.9)
    ax[j, 1].set_ylabel("PC proj.")
    ax[j, 1].set_title(f"corr {corr[j]:.2f}", fontsize=7.5)
ax[1, 0].set_xlabel("time (s)")
ax[1, 1].set_xlabel("time (s)")
plotlib.save(f, "p1_pca_separation")
print(f"correlation with truth: {corr.round(3)}")
print(f"SNR: {bss.snr_db(S, P).round(1)} dB")

## Slide: Rank-1 reconstruction drops a source

PCA reconstruction with 1 component = the data projected on the biggest variance direction:
the second source is gone from microphone 1 but leaks into the reconstruction.

In [ ]:
pca1 = PCA(n_components=1).fit(X.T)
X1 = pca1.inverse_transform(pca1.transform(X.T)).T
f, ax = plt.subplots(2, 2, figsize=(plotlib.FULL, 2.3), sharex=True)
for j in range(2):
    ax[j, 0].plot(t, X[j], color=GREY, lw=0.9)
    ax[j, 0].set_ylabel(f"$x_{j+1}$")
    ax[j, 1].plot(t, X1[j], color=BLUE, lw=0.9)
    ax[j, 1].set_ylabel("1 PC")
ax[0, 0].set_title("microphone 1", fontsize=7.5)
ax[0, 1].set_title("reconstruction", fontsize=7.5)
ax[1, 0].set_xlabel("time (s)")
ax[1, 1].set_xlabel("time (s)")
plotlib.save(f, "p1_rank1")
print(f"relative reconstruction error (1 PC): {np.linalg.norm(X - X1) / np.linalg.norm(X):.3f}")
print(f"relative error (2 PCs): {np.linalg.norm(X - PCA(2).fit(X.T).inverse_transform(PCA(2).fit(X.T).transform(X.T)).T) / np.linalg.norm(X):.3f}")

## Slide: The scree plot decides how many components

PCA on spambase (57 features, standardised): cumulative explained variance and the scree.
90 % of the variance sits in 30 of 57 components; the knee is a decision, not a law.

In [ ]:
Xspam, _, names = data.load_table("spambase", "spam")
mu, sd = Xspam.mean(0), Xspam.std(0)
Xs = (Xspam - mu) / sd
pca_s = PCA().fit(Xs)
evr = pca_s.explained_variance_ratio_
f, ax = plt.subplots(figsize=(plotlib.COL, 2.1))
ax.plot(np.arange(1, 58), evr, color=BLUE, lw=1.2)
ax.plot(np.arange(1, 58), np.cumsum(evr), color=ORANGE, lw=1.2)
ax.set_xlabel("component")
ax.set_ylabel("variance share")
ax.legend(["individual", "cumulative"], fontsize=7)
plotlib.save(f, "p1_scree")
k90 = int(np.searchsorted(np.cumsum(evr), 0.9) + 1)
print(f"components for 90 % variance: {k90}")
print(f"first three shares: {evr[:3].round(3)}")

## Slide: Components of digits are averages of digits

PCA on digits (8 x 8 images): the mean image, PC1-PC4 and the rank-r reconstruction of one digit.
Components look like blurry digits, not like parts.

In [ ]:
Xd, _, _ = data.load_table("digits", "target")
Xd = Xd / Xd.max()
pca_d = PCA().fit(Xd)
digit = Xd[0]
f, ax = plt.subplots(2, 4, figsize=(plotlib.FULL, 2.7))
ims = [digit.reshape(8, 8)] + [pca_d.mean_.reshape(8, 8)] + [pca_d.components_[j].reshape(8, 8) for j in range(2)]
titles = ["a digit", "mean image", "PC1", "PC2"]
for a, im, ti in zip(ax[0], ims, titles):
    a.imshow(im, cmap="viridis")
    a.set_title(ti, fontsize=7.5)
    a.set_xticks([])
    a.set_yticks([])
    a.set_aspect("equal")
for a, r in zip(ax[1], [16, 8, 4]):
    rec = pca_d.mean_ + (pca_d.transform(digit[None]) @ pca_d.components_[:, :r] @ pca_d.components_[:r])[:, :64][0]
    a.imshow(rec.reshape(8, 8), cmap="viridis")
    a.set_title(f"{r} components", fontsize=7.5)
    a.set_xticks([])
    a.set_yticks([])
    a.set_aspect("equal")
ax[1, 3].axis("off")
plotlib.save(f, "p1_digits")

## Sweep and failure cases (more than the slides)

1. Scaling matters: PCA on raw spambase puts almost all variance in the huge-count features.
2. PCA on a Gaussian cloud: every direction is equal, the scree is flat, nothing separates.

In [ ]:
slide("Scaling failure")
pca_raw = PCA().fit(Xspam)
print(f"raw features, PC1 share: {pca_raw.explained_variance_ratio_[0]:.3f}")
print(f"standardised, PC1 share: {PCA().fit(Xs).explained_variance_ratio_[0]:.3f}")

slide("Gaussian cloud, flat scree")
g = np.random.default_rng(SEED).normal(size=(500, 6))
evr_g = PCA().fit(g).explained_variance_ratio_
print(f"explained variance ratio: {evr_g.round(3)}")

In [ ]:
print(f"total: {pl.DataFrame({'cell': ['A'], 's': [0]}).height and ''}demo 09a done")